In [ ]:
%uv pip install scikit-learn ipywidgets jupyterlab

In [1]:
!mkdir /root/checkpoints

# Transformer Code

In [32]:
import math

import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int = 32, max_len: int = 1500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        
        return x

class IOHTransformer(nn.Module):
    def __init__(self, model_dim=64, num_heads=8):
        super(IOHTransformer, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim=model_dim, num_heads=num_heads, batch_first=True)
        self.layer_norm_attn = nn.LayerNorm(model_dim)

        self.ffn = nn.Sequential(
            nn.Linear(model_dim, model_dim * 4),
            nn.ReLU(),
            nn.Linear(model_dim * 4, model_dim)
        )
        self.layer_norm_ffn = nn.LayerNorm(model_dim)

    def forward(self, x):
        # x shape: (Batch, seq_len, input_dim)
        attn_output, _ = self.multihead_attn(x, x, x)  # (Batch, seq_len, model_dim)
        x = self.layer_norm_attn(x + attn_output)  # Residual connection + LayerNorm

        ffn_output = self.ffn(x)  # (Batch, seq_len, model_dim)
        x = self.layer_norm_ffn(x + ffn_output)  # Residual connection + LayerNorm

        return x

class IOHPredictor(nn.Module):
    def __init__(self, input_dim = 4, model_dim_1 = 32, model_dim_2 = 64, num_heads = 4):
        super(IOHPredictor, self).__init__()
        self.input_projection_1 = nn.Linear(input_dim, model_dim_1)
        self.pos_embed = PositionalEncoding(d_model=model_dim_1)
        self.transformer_1 = IOHTransformer(model_dim=model_dim_1, num_heads=num_heads)

        self.conv1d = nn.Conv1d(model_dim_1, model_dim_2, kernel_size=5, stride=2)

        # self.input_projection_2 = nn.Linear(model_dim_1, model_dim_2)
        self.transformer_2 = IOHTransformer(model_dim=model_dim_2, num_heads=num_heads)
        self.output_projection = nn.Sequential(
            nn.Linear(model_dim_2+5, model_dim_2 // 2),
            nn.GELU(),
            nn.Linear(model_dim_2 // 2, 1)
        )
    
    def forward(self, x_seq, x_static):
        x = self.input_projection_1(x_seq) # (B, 900, 4)
        x = self.pos_embed(x)  # (B, 900, 4) -> (B, 900, model_dim)
        x = self.transformer_1(x)  # (Batch, 900, 32)

        # x = self.input_projection_2(x)
        x = self.conv1d(x.transpose(1, 2)).transpose(1, 2)
        x = torch.nn.functional.gelu(x)
        x = self.transformer_2(x)  # (Batch, 900, 64)

        x = torch.mean(x, dim=1)  # (B, 64)
        x = torch.concat([x, x_static], dim=-1)  # (B, 64 + 5)

        output = self.output_projection(x)  # (B, 1)
        return output.squeeze(-1)  # (B,)

if __name__ == "__main__":
    model = IOHPredictor(input_dim=4, model_dim_1=32, model_dim_2=64, num_heads=4)
    dummy_seq = torch.randn(2, 900, 4)  # (Batch, seq_len, input_dim)
    dummy_static = torch.randn(2, 5)# (Batch, static_dim)

    print("Parameters: ", sum(p.numel() for p in model.parameters()))

    output = model(dummy_seq, dummy_static)
    print("Output shape:", output.shape)

Parameters:  75425
Output shape: torch.Size([2])


# Mamba Model Code

In [33]:
import torch
import torch.nn as nn

try:
    from mamba_ssm import Mamba
except ImportError:
    raise ImportError("Please install mamba_ssm and causal-conv1d to use this model.")

class IOHMambaPredictor(nn.Module):
    def __init__(self, input_dim=4, model_dim_1=32, model_dim_2=64):
        super(IOHMambaPredictor, self).__init__()
        
        # 1. Initial Projection 
        self.input_projection_1 = nn.Linear(input_dim, model_dim_1)
        self.norm_1 = nn.LayerNorm(model_dim_1)
        
        # 2. Mamba Block 1
        self.mamba_1 = Mamba(
            d_model=model_dim_1,
            d_state=16,
            d_conv=4,
            expand=2
        )

        # 3. Projection to larger dimension
        self.input_projection_2 = nn.Linear(model_dim_1, model_dim_2)
        self.norm_2 = nn.LayerNorm(model_dim_2)
        
        # 4. Mamba Block 2
        self.mamba_2 = Mamba(
            d_model=model_dim_2,
            d_state=16,
            d_conv=4,
            expand=2
        )
        
        # 5. Final Output Projection (Late Fusion - Identical to Transformer)
        self.output_projection = nn.Sequential(
            nn.Linear(model_dim_2 + 5, model_dim_2 // 2),
            nn.ELU(),
            nn.Linear(model_dim_2 // 2, 1)
        )
    
    def forward(self, x_seq, x_static):
        # Sequence: (B, 900, 4)
        x = self.input_projection_1(x_seq) # (B, 900, 64)
        x = self.norm_1(x)
        x = self.mamba_1(x)                # (B, 900, 64)
        
        x = self.input_projection_2(x)     # (B, 900, 64)
        x = self.norm_2(x)
        x = self.mamba_2(x)                # (B, 900, 64)

        # 1. Pool the time-series sequence
        x = torch.mean(x, dim=1)           # (B, 64)

        # 2. Concatenate static demographics
        x = torch.concat([x, x_static], dim=-1)  # (B, 69)

        # 3. Final prediction
        output = self.output_projection(x) # (B, 1)
        
        return output.squeeze(-1)          # (B)

if __name__ == "__main__":
    # Mamba requires CUDA. It will crash on CPU due to the custom kernels.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = IOHMambaPredictor().to(device)
    
    # Calculate trainable parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Mamba Model Parameters: {total_params:,}")
    
    if device.type == "cuda":
        dummy_seq = torch.randn(32, 900, 4).to(device)
        dummy_static = torch.randn(32, 5).to(device)
        
        out = model(dummy_seq, dummy_static)
        print(f"Output Shape: {out.shape} (Expected: [32])")
    else:
        print("Warning: CUDA not detected. Mamba forward pass skipped.")

Mamba Model Parameters: 47,297
Output Shape: torch.Size([32]) (Expected: [32])


# IOH Dataset

In [34]:
import os
from typing import Dict
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm

class IOHDataset(Dataset):
    def __init__(self, data_dir: str, manifest: Dict[int, int]):
        super().__init__()
        self.data_dir = data_dir
        
        # 1. Calculate total windows to pre-allocate exact memory
        total_windows = sum(manifest.values())
        print(f"Pre-allocating RAM for {total_windows} windows...")

        # 2. Pre-allocate massive empty tensors (Zero memory spike!)
        self.X_seq = torch.empty((total_windows, 900, 4), dtype=torch.float32)
        self.X_static = torch.empty((total_windows, 5), dtype=torch.float32)
        self.Y = torch.empty((total_windows,), dtype=torch.long)

        # 3. Fill the tensors directly from disk
        current_idx = 0
        for case_id in tqdm(sorted(manifest.keys())):
            n_windows = manifest[case_id]
            if n_windows == 0:
                continue
                
            path = os.path.join(data_dir, f"case_{case_id}.pt")
            data = torch.load(path, weights_only=True)
            assert data["X_seq"].shape[0] == n_windows, \
                f"case_{case_id}: manifest says {n_windows} windows but file has {data['X_seq'].shape[0]}"
            
            # Slot the data into the pre-allocated block
            end_idx = current_idx + n_windows
            self.X_seq[current_idx:end_idx] = data["X_seq"]
            self.X_static[current_idx:end_idx] = data["X_static"]
            self.Y[current_idx:end_idx] = data["Y"]
            
            current_idx = end_idx

        print(f"Dataset successfully loaded into RAM. Size: {self.X_seq.element_size() * self.X_seq.nelement() / 1e9:.2f} GB")

    def __len__(self) -> int:
        return len(self.Y)

    def __getitem__(self, idx: int):
        return self.X_seq[idx], self.X_static[idx], self.Y[idx]

# Configuration

In [35]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
)
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_DIR   = "/mnt/dataforioh/processed_data_denovo/train"   # TODO: confirm path
TEST_DIR    = "/mnt/dataforioh/processed_data_denovo/test"    # TODO: confirm path
OUTPUT_DIR  = "/mnt/dataforioh/processed_data_denovo"         # TODO: confirm path

# Sampling and windowing constants — must match preprocessing pipeline exactly
SAMPLING_HZ         = 0.5        # 0.5 Hz = one sample every 2 seconds
SAMPLES_PER_MIN     = SAMPLING_HZ * 60   # 30 samples per minute
OBS_WINDOW_SAMPLES  = 900        # 30 minutes
LEAD_GAP_SAMPLES    = 150        # 5 minutes
PRED_WINDOW_SAMPLES = 300        # 10 minutes
STEP_SIZE_SAMPLES   = 30         # 1 minute slide step

# Bootstrap settings
N_BOOTSTRAP         = 1000
BOOTSTRAP_SEED      = 42
CI_ALPHA            = 0.05       # 95% confidence interval

# Random guessing AUPRC baseline — positive prevalence on test set
RANDOM_AUPRC_BASELINE = 0.0657   # 6096 / 92751

# Validation split seed — must match training scripts exactly
VAL_SPLIT_SEED = 42

# ── Model configurations ──────────────────────────────────────────────────────
# Fill in checkpoint paths. List all three seed paths per model.
# For models trained on D2 (input_dim=2) or D6 (input_dim=2), set input_dim accordingly.
# For M5 (no covariates), the architecture is identical but covariates are zeroed at inference.

MODEL_CONFIGS = {
    "M1_Mamba_D1": {
        "label":        "Mamba — Full signals (D1)",
        "input_dim":    4,
        "no_covariates": False,
        "drug_only":    False,
        "vitals_only":  False,
        "transformer": False,
        "checkpoints": [
            "/root/checkpoints/PM_47k_D1_denovo_42.pth",   # TODO: seed 42  e.g. "/mnt/dataforioh/checkpoints/PM_47k_D1_seed42.pth"
            "/root/checkpoints/PM_47k_D1_denovo_123.pth",   # TODO: seed 123
            "/root/checkpoints/PM_47k_D1_denovo_7.pth",   # TODO: seed 7
        ],
    },
    "M2_Mamba_D2": {
        "label":        "Mamba — Vitals only (D2)",
        "input_dim":    2,
        "no_covariates": False,
        "drug_only":    False,
        "vitals_only":  True,
        "transformer": False,
        "checkpoints": [
            "/root/checkpoints/PM_47k_D2_MAPHRonly_42.pth",   # TODO: seed 42
            "/root/checkpoints/PM_47k_D2_MAPHRonly_123.pth",   # TODO: seed 123
            "/root/checkpoints/PM_47k_D2_MAPHRonly_7.pth",   # TODO: seed 7
        ],
    },
    "M4_Mamba_D4": {
        "label":        "Mamba — Contaminated (D4)",
        "input_dim":    4,
        "no_covariates": False,
        "drug_only":    False,
        "vitals_only":  False,
        "transformer": False,
        "checkpoints": [
            "/root/checkpoints/PM_47k_D4_contaminated_42.pth",   # TODO: seed 42
            "/root/checkpoints/PM_47k_D4_contaminated_123.pth",   # TODO: seed 123
            "/root/checkpoints/PM_47k_D4_contaminated_7.pth",   # TODO: seed 7
        ],
    },
    "M5_Mamba_D5": {
        "label":        "Mamba — No covariates (D5)",
        "input_dim":    4,
        "no_covariates": True,
        "drug_only":    False,
        "vitals_only":  False,
        "transformer": False,
        "checkpoints": [
            "/root/checkpoints/PM_47k_D5_NoCovariates_42.pth",   # TODO: seed 42
            "/root/checkpoints/PM_47k_D5_NoCovariates_123.pth",   # TODO: seed 123
            "/root/checkpoints/PM_47k_D5_NoCovariates_7.pth",   # TODO: seed 7
        ],
    },
    "M6_Mamba_D6": {
        "label":        "Mamba — Drugs only (D6)",
        "input_dim":    2,
        "no_covariates": False,
        "drug_only":    True,
        "vitals_only":  False,
        "transformer": False,
        "checkpoints": [
            "/root/checkpoints/PM_47k_D6_Drugsonly_42.pth",   # TODO: seed 42
            "/root/checkpoints/PM_47k_D6_Drugsonly_123.pth",   # TODO: seed 123
            "/root/checkpoints/PM_47k_D6_Drugsonly_7.pth",   # TODO: seed 7
        ],
    },
    "T1_Transformer_D1": {
        "label":        "Transformer — Full signals (D1)",
        "input_dim":    4,
        "no_covariates": False,
        "drug_only":    False,
        "vitals_only":  False,
        "transformer": True,
        "checkpoints": [
            "/root/checkpoints/PT_75k_D1_denovo_42.pth",   # TODO: seed 42
            "/root/checkpoints/PT_75k_D1_denovo_123.pth",   # TODO: seed 123
            "/root/checkpoints/PT_75k_D1_denovo_7.pth",   # TODO: seed 7
        ],
    },
}

# DeLong comparison pairs — (model_A_key, model_B_key, description)
DELONG_PAIRS = [
    ("M1_Mamba_D1",      "M2_Mamba_D2",      "M1 vs M2: effect of drug CE"),
    ("M1_Mamba_D1",      "M6_Mamba_D6",      "M1 vs M6: effect of vitals"),
    ("M1_Mamba_D1",      "M5_Mamba_D5",      "M1 vs M5: effect of covariates"),
    ("M1_Mamba_D1",      "M4_Mamba_D4",      "M1 vs M4: contamination effect"),
    ("M1_Mamba_D1",      "T1_Transformer_D1","M1 vs T1: Mamba vs Transformer"),
]

# Data Loading

In [36]:
def build_loaders(batch_size=256):
    """
    Build validation loader (from training manifest) and test loader.
    Validation loader is used exclusively for threshold selection on M1.
    Test loader is used for all reported metrics.
    """
    meta_path = os.path.join(OUTPUT_DIR, "pipeline_meta.pkl")
    with open(meta_path, "rb") as f:
        meta = pickle.load(f)

    full_train_manifest = meta["manifest"]["train"]
    test_manifest       = meta["manifest"]["test"]

    train_case_ids = list(full_train_manifest.keys())
    _, val_ids = train_test_split(
        train_case_ids, test_size=0.2, random_state=VAL_SPLIT_SEED
    )
    val_manifest = {cid: full_train_manifest[cid] for cid in val_ids}

    val_ds   = IOHDataset(TRAIN_DIR, val_manifest)
    test_ds  = IOHDataset(TEST_DIR,  test_manifest)

    val_loader  = DataLoader(val_ds,  batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

    print(f"[INFO] Val  windows : {len(val_ds)}")
    print(f"[INFO] Test windows : {len(test_ds)}")

    return val_loader, test_loader

# Model Inference

In [39]:
def build_model(config):
    """Instantiate a fresh model. Extend this if Transformer needs different args."""
    if config["transformer"]:
        return IOHPredictor(input_dim=config["input_dim"], model_dim_1=32, model_dim_2=64, num_heads=4)
    else:
        return IOHMambaPredictor(input_dim=config["input_dim"], model_dim_1=32, model_dim_2=64)


def run_inference(model, loader, config):
    """
    Run inference on a dataloader and return (probabilities, labels) as numpy arrays.
    Handles vitals_only, drug_only, and no_covariates ablation variants.
    """
    model.eval()
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for X_seq, X_static, labels in tqdm(loader):
            X_seq    = X_seq.to(DEVICE)
            labels   = labels.float().to(DEVICE)

            # --- Signal selection based on model config ---
            if config["vitals_only"]:
                # D2: keep only MAP and HR (channels 0 and 1)
                X_seq = X_seq[:, :, :2]
            elif config["drug_only"]:
                # D6: keep only Propofol CE and Remifentanil CE (channels 2 and 3)
                X_seq = X_seq[:, :, 2:]

            # --- Covariate handling ---
            if config["no_covariates"]:
                # D5: zero out covariates — model sees no static information
                X_static = torch.zeros(X_seq.size(0), 5).to(DEVICE)
            else:
                X_static = X_static.to(DEVICE)

            logits = model(X_seq, X_static)
            probs  = torch.sigmoid(logits)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_probs), np.array(all_labels)


def get_mean_probs_across_seeds(config, loader):
    """
    Load each seed's checkpoint, run inference, and return:
        - per_seed_probs : list of 3 numpy arrays, one per seed
        - labels         : numpy array (identical across seeds, taken from seed 0)
    """
    per_seed_probs = []
    labels_out     = None

    for ckpt_path in config["checkpoints"]:
        assert ckpt_path != "", (
            f"Checkpoint path not filled in for {config['label']}. "
            f"Please fill in MODEL_CONFIGS."
        )
        model = build_model(config)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
        model.to(DEVICE)

        probs, labels = run_inference(model, loader, config)
        per_seed_probs.append(probs)

        if labels_out is None:
            labels_out = labels

    return per_seed_probs, labels_out

# Bootstrap CI

In [38]:
def bootstrap_metric(y_true, y_score, metric_fn, n_bootstrap=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    """
    Compute a metric and its 95% CI via stratified bootstrap resampling.

    Returns
    -------
    (point_estimate, ci_lower, ci_upper)
    """
    rng = np.random.default_rng(seed)
    point = metric_fn(y_true, y_score)

    bootstrapped = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(y_true), size=len(y_true))
        y_true_b  = y_true[idx]
        y_score_b = y_score[idx]

        # Skip degenerate bootstrap samples with only one class
        if len(np.unique(y_true_b)) < 2:
            continue
        bootstrapped.append(metric_fn(y_true_b, y_score_b))

    bootstrapped = np.array(bootstrapped)
    ci_lower = np.percentile(bootstrapped, 100 * CI_ALPHA / 2)
    ci_upper = np.percentile(bootstrapped, 100 * (1 - CI_ALPHA / 2))

    return point, ci_lower, ci_upper

# Threshold Selection

In [37]:
def select_threshold_youden(y_true, y_score):
    """
    Select the optimal decision threshold using Youden's J statistic.
    Youden's J = Sensitivity + Specificity - 1.
    Maximising J balances sensitivity and specificity optimally.

    Returns
    -------
    float : optimal threshold
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    specificity = 1 - fpr
    j_scores    = tpr + specificity - 1
    best_idx    = np.argmax(j_scores)
    optimal_threshold = thresholds[best_idx]
    print(f"[INFO] Youden's J optimal threshold: {optimal_threshold:.4f} "
          f"(Sensitivity={tpr[best_idx]:.3f}, Specificity={specificity[best_idx]:.3f})")
    return float(optimal_threshold)


def compute_threshold_metrics(y_true, y_score, threshold):
    """
    Compute Sensitivity, Specificity, PPV, NPV, F1 at a fixed threshold.

    Returns
    -------
    dict with keys: sensitivity, specificity, ppv, npv, f1
    """
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv         = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    f1          = f1_score(y_true, y_pred, zero_division=0)

    return {
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv":         ppv,
        "npv":         npv,
        "f1":          f1,
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    }

# Clinical Utility Metrics

In [40]:
def compute_lead_time(y_true, y_score, threshold):
    """
    Compute mean lead time in minutes for true positive predictions.

    Each positive window (label=1) was generated 5 minutes before an IOH event
    by definition of the pipeline. However, within that, the actual lead time
    depends on where in the prediction window the IOH event first occurs.

    Since we do not have per-window IOH onset offsets stored, we use the
    conservative estimate that every TP prediction leads the IOH event by the
    full lead gap plus half the prediction window on average:
        lead gap (5 min) + prediction window / 2 (5 min) = 10 min median estimate.

    If you have stored IOH onset offsets per window, replace this function
    with a direct calculation using those offsets.

    For a data-driven estimate, we report the distribution across TP windows
    using the known pipeline parameters.

    Returns
    -------
    dict with keys: mean_lead_time_min, std_lead_time_min, n_true_positives
    """
    y_pred = (y_score >= threshold).astype(int)
    tp_mask = (y_true == 1) & (y_pred == 1)
    n_tp = int(tp_mask.sum())

    if n_tp == 0:
        return {"mean_lead_time_min": 0.0, "std_lead_time_min": 0.0, "n_true_positives": 0}

    # Pipeline-derived lead time bounds:
    # Minimum: lead gap only = 5 minutes (IOH occurs at start of prediction window)
    # Maximum: lead gap + full prediction window = 15 minutes (IOH at end of window)
    # Each TP window was generated with STEP_SIZE_SAMPLES = 30 samples = 1 minute steps
    # Conservative uniform distribution estimate across prediction window

    LEAD_GAP_MIN    = LEAD_GAP_SAMPLES    / SAMPLES_PER_MIN   # 5.0 min
    PRED_WINDOW_MIN = PRED_WINDOW_SAMPLES / SAMPLES_PER_MIN   # 10.0 min

    # Uniform distribution over [5, 15] minutes → mean = 10 min, std = sqrt((10²)/12) ≈ 2.89 min
    mean_lead = LEAD_GAP_MIN + PRED_WINDOW_MIN / 2.0
    std_lead  = PRED_WINDOW_MIN / np.sqrt(12)

    print(f"[NOTE] Lead time is estimated from pipeline parameters (uniform distribution "
          f"over [{LEAD_GAP_MIN:.0f}, {LEAD_GAP_MIN + PRED_WINDOW_MIN:.0f}] min). "
          f"For exact per-window lead times, store IOH onset offsets during preprocessing.")

    return {
        "mean_lead_time_min": mean_lead,
        "std_lead_time_min":  std_lead,
        "n_true_positives":   n_tp,
    }


def compute_false_alarm_rate(y_true, y_score, threshold, total_surgery_hours):
    """
    Compute false alarm rate as number of false positive predictions per hour of surgery.

    Parameters
    ----------
    y_true              : numpy array of true labels
    y_score             : numpy array of predicted probabilities
    threshold           : decision threshold
    total_surgery_hours : total hours of surgery covered by the test set
                          Computed as: n_windows * step_size_seconds / 3600

    Returns
    -------
    dict with keys: false_alarms_per_hour, n_false_positives, total_hours
    """
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    false_alarms_per_hour = fp / total_surgery_hours if total_surgery_hours > 0 else 0.0

    return {
        "false_alarms_per_hour": false_alarms_per_hour,
        "n_false_positives":     int(fp),
        "total_surgery_hours":   total_surgery_hours,
    }


def estimate_total_surgery_hours(n_test_windows):
    """
    Estimate total surgery hours covered by the test set.

    Each window slides by STEP_SIZE_SAMPLES at SAMPLING_HZ.
    Total time = n_windows * step_size_in_seconds / 3600.

    This is an upper bound — it counts overlapping windows once each.
    The true unique coverage is less due to sliding window overlap,
    but this is the standard convention for false alarm rate reporting.
    """
    step_size_seconds = STEP_SIZE_SAMPLES / SAMPLING_HZ   # 60 seconds = 1 minute
    total_seconds     = n_test_windows * step_size_seconds
    return total_seconds / 3600.0

# Delong Test

In [41]:
def delong_roc_variance(ground_truth, predictions):
    """
    Computes ROC AUC variance using the DeLong method.
    Based on: DeLong et al. (1988) Biometrics 44:837-845.
    Implementation follows Sun & Xu (2014) Academic Radiology.
    """
    order      = (-predictions).argsort()
    label_1_count = int(ground_truth.sum())
    label_0_count = len(ground_truth) - label_1_count

    predictions_sorted_transposed = predictions[np.newaxis, order]
    ground_truth_sorted           = ground_truth[order]

    aucs, delongcov = _fastDeLong(predictions_sorted_transposed, label_1_count)
    return aucs, delongcov


def _fastDeLong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m

    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)

    for r in range(k):
        tx[r, :] = _compute_midrank(positive_examples[r, :])
        ty[r, :] = _compute_midrank(negative_examples[r, :])
        tz[r, :] = _compute_midrank(predictions_sorted_transposed[r, :])

    aucs = (tz[:, :m].sum(axis=1) - tx.sum(axis=1)) / (m * n)

    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m

    sx = np.cov(v01) if v01.shape[1] > 1 else np.array([[np.var(v01)]])
    sy = np.cov(v10) if v10.shape[1] > 1 else np.array([[np.var(v10)]])

    delongcov = sx / m + sy / n
    return aucs, delongcov


def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2


def delong_test(y_true, y_score_a, y_score_b):
    """
    Two-sided DeLong test comparing AUROC of two models on the same test set.

    Returns
    -------
    (auc_a, auc_b, z_stat, p_value)
    """
    m = int(y_true.sum())
    n = len(y_true) - m
    order = (-y_true).argsort()

    y_true_sorted = y_true[order]

    score_ab = np.vstack([y_score_a[order], y_score_b[order]])
    aucs, cov = _fastDeLong(score_ab, m)

    auc_a, auc_b = aucs
    var_a  = cov[0, 0]
    var_b  = cov[1, 1]
    cov_ab = cov[0, 1]

    se    = np.sqrt(var_a + var_b - 2 * cov_ab)
    z     = (auc_a - auc_b) / se if se > 0 else 0.0
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))

    return float(auc_a), float(auc_b), float(z), float(p_val)

# Formatting Results

In [42]:
def print_separator(char="=", width=80):
    print(char * width)


def print_model_results(model_key, config, results):
    print_separator()
    print(f"  {config['label']}")
    print_separator()

    r = results
    print(f"  AUROC : {r['auroc']:.4f}  (95% CI: {r['auroc_ci_lo']:.4f} – {r['auroc_ci_hi']:.4f})")
    print(f"  AUPRC : {r['auprc']:.4f}  (95% CI: {r['auprc_ci_lo']:.4f} – {r['auprc_ci_hi']:.4f})")
    print(f"  Random AUPRC baseline: {RANDOM_AUPRC_BASELINE:.4f}")
    print(f"  AUPRC lift over random: {r['auprc'] / RANDOM_AUPRC_BASELINE:.2f}x")
    print()
    print(f"  ── Threshold metrics (threshold = {r['threshold']:.4f}) ──")
    print(f"  Sensitivity : {r['sensitivity']:.4f}")
    print(f"  Specificity : {r['specificity']:.4f}")
    print(f"  PPV         : {r['ppv']:.4f}")
    print(f"  NPV         : {r['npv']:.4f}")
    print(f"  F1-Score    : {r['f1']:.4f}")
    print(f"  TP={r['tp']}  TN={r['tn']}  FP={r['fp']}  FN={r['fn']}")
    print()
    print(f"  ── Clinical utility ──")
    print(f"  True positive predictions   : {r['n_true_positives']}")
    print(f"  Mean lead time              : {r['mean_lead_time_min']:.1f} ± {r['std_lead_time_min']:.1f} min")
    print(f"  False alarm rate            : {r['false_alarms_per_hour']:.2f} alarms/hour")
    print(f"  Total surgery hours (test)  : {r['total_surgery_hours']:.1f} h")
    print()

    # Per-seed breakdown
    print(f"  ── Per-seed AUROC / AUPRC ──")
    for i, (au, ap) in enumerate(zip(r["seed_aurocs"], r["seed_auprcs"])):
        print(f"  Seed {i+1}: AUROC={au:.4f}  AUPRC={ap:.4f}")
    print(f"  Mean AUROC: {np.mean(r['seed_aurocs']):.4f} ± {np.std(r['seed_aurocs']):.4f}")
    print(f"  Mean AUPRC: {np.mean(r['seed_auprcs']):.4f} ± {np.std(r['seed_auprcs']):.4f}")


def print_delong_results(delong_results, n_comparisons):
    print_separator()
    print("  DeLong Test Results")
    print(f"  Bonferroni-corrected significance threshold: "
          f"p < {CI_ALPHA / n_comparisons:.4f} (α={CI_ALPHA}, n={n_comparisons})")
    print_separator()

    for desc, auc_a, auc_b, z, p, p_corrected, significant in delong_results:
        sig_marker = "***" if significant else "n.s."
        print(f"  {desc}")
        print(f"    AUROC A={auc_a:.4f}  AUROC B={auc_b:.4f}  "
              f"z={z:.3f}  p={p:.4f}  p_bonf={p_corrected:.4f}  {sig_marker}")
        print()

# Main Pipeline

In [43]:
def main():
    print_separator()
    print("  IOH Prediction — Step 5 Metrics Evaluation")
    print_separator()

    # ── Build data loaders ──
    val_loader, test_loader = build_loaders()
    n_test_windows      = sum(1 for _ in test_loader.dataset)
    total_surgery_hours = estimate_total_surgery_hours(n_test_windows)
    print(f"[INFO] Test windows: {n_test_windows}")
    print(f"[INFO] Estimated total surgery hours: {total_surgery_hours:.1f} h\n")

    # ── Step A: Get M1 validation probabilities for threshold selection ──
    print("[STEP A] Selecting threshold from M1 validation set using Youden's J...")
    m1_config = MODEL_CONFIGS["M1_Mamba_D1"]

    # Use the first seed checkpoint for threshold selection
    # (threshold is fixed and applied uniformly across all models)
    model_m1 = build_model(m1_config)
    model_m1.load_state_dict(
        torch.load(m1_config["checkpoints"][0], map_location=DEVICE, weights_only=True)
    )
    model_m1.to(DEVICE)
    val_probs_m1, val_labels_m1 = run_inference(model_m1, val_loader, m1_config)
    GLOBAL_THRESHOLD = select_threshold_youden(val_labels_m1, val_probs_m1)
    print(f"[INFO] Global threshold fixed at: {GLOBAL_THRESHOLD:.4f}\n")

    # ── Step B: Evaluate all models on test set ──
    all_results = {}
    # Store mean probs per model for DeLong test
    mean_probs_store = {}
    test_labels_store = None

    for model_key, config in MODEL_CONFIGS.items():
        print(f"\n[EVALUATING] {config['label']} ...")

        per_seed_probs, test_labels = get_mean_probs_across_seeds(config, test_loader)

        if test_labels_store is None:
            test_labels_store = test_labels

        # Mean probability across seeds — used for bootstrap CI and DeLong
        mean_probs  = np.mean(per_seed_probs, axis=0)
        mean_probs_store[model_key] = mean_probs

        # Per-seed individual metrics
        seed_aurocs = [roc_auc_score(test_labels, p) for p in per_seed_probs]
        seed_auprcs = [average_precision_score(test_labels, p) for p in per_seed_probs]

        # Bootstrap CI on mean probs
        auroc, auroc_lo, auroc_hi = bootstrap_metric(
            test_labels, mean_probs, roc_auc_score
        )
        auprc, auprc_lo, auprc_hi = bootstrap_metric(
            test_labels, mean_probs, average_precision_score
        )

        # Threshold-based metrics
        tm = compute_threshold_metrics(test_labels, mean_probs, GLOBAL_THRESHOLD)

        # Clinical utility
        lt = compute_lead_time(test_labels, mean_probs, GLOBAL_THRESHOLD)
        fa = compute_false_alarm_rate(
            test_labels, mean_probs, GLOBAL_THRESHOLD, total_surgery_hours
        )

        all_results[model_key] = {
            # Discrimination
            "auroc":          auroc,
            "auroc_ci_lo":    auroc_lo,
            "auroc_ci_hi":    auroc_hi,
            "auprc":          auprc,
            "auprc_ci_lo":    auprc_lo,
            "auprc_ci_hi":    auprc_hi,
            # Threshold
            "threshold":      GLOBAL_THRESHOLD,
            **tm,
            # Clinical utility
            "mean_lead_time_min":  lt["mean_lead_time_min"],
            "std_lead_time_min":   lt["std_lead_time_min"],
            "n_true_positives":    lt["n_true_positives"],
            "false_alarms_per_hour": fa["false_alarms_per_hour"],
            "n_false_positives":   fa["n_false_positives"],
            "total_surgery_hours": fa["total_surgery_hours"],
            # Per-seed
            "seed_aurocs": seed_aurocs,
            "seed_auprcs": seed_auprcs,
        }

    # ── Step C: Print all model results ──
    print("\n\n")
    print_separator("═")
    print("  FULL RESULTS")
    print_separator("═")
    for model_key, config in MODEL_CONFIGS.items():
        print_model_results(model_key, config, all_results[model_key])

    # ── Step D: DeLong tests ──
    print("\n")
    n_comparisons = len(DELONG_PAIRS)
    bonferroni_threshold = CI_ALPHA / n_comparisons

    delong_results = []
    for key_a, key_b, desc in DELONG_PAIRS:
        probs_a = mean_probs_store[key_a]
        probs_b = mean_probs_store[key_b]
        auc_a, auc_b, z, p = delong_test(test_labels_store, probs_a, probs_b)
        p_corrected = min(p * n_comparisons, 1.0)   # Bonferroni correction
        significant = p_corrected < CI_ALPHA
        delong_results.append((desc, auc_a, auc_b, z, p, p_corrected, significant))

    print_delong_results(delong_results, n_comparisons)

    # ── Step E: Summary table ──
    print_separator("═")
    print("  SUMMARY TABLE (for paper)")
    print_separator("═")
    header = f"{'Model':<35} {'AUROC':>12} {'AUPRC':>12} {'Sens':>7} {'Spec':>7} {'PPV':>7} {'NPV':>7} {'F1':>7}"
    print(header)
    print("-" * len(header))
    for model_key, config in MODEL_CONFIGS.items():
        r = all_results[model_key]
        row = (
            f"{config['label']:<35} "
            f"{r['auroc']:.4f} ({r['auroc_ci_lo']:.3f}–{r['auroc_ci_hi']:.3f})  "
            f"{r['auprc']:.4f} ({r['auprc_ci_lo']:.3f}–{r['auprc_ci_hi']:.3f})  "
            f"{r['sensitivity']:.3f}  "
            f"{r['specificity']:.3f}  "
            f"{r['ppv']:.3f}  "
            f"{r['npv']:.3f}  "
            f"{r['f1']:.3f}"
        )
        print(row)
    print(f"\nRandom guessing AUPRC baseline: {RANDOM_AUPRC_BASELINE:.4f}")
    print(f"Global threshold (Youden's J on M1 val set): {GLOBAL_THRESHOLD:.4f}")
    print(f"Bootstrap iterations: {N_BOOTSTRAP}")
    print(f"Bonferroni threshold: p < {bonferroni_threshold:.4f} for {n_comparisons} comparisons")

    return all_results


if __name__ == "__main__":
    results = main()

  IOH Prediction — Step 5 Metrics Evaluation
Pre-allocating RAM for 20172 windows...


  0%|          | 0/545 [00:00<?, ?it/s]

Dataset successfully loaded into RAM. Size: 0.29 GB
Pre-allocating RAM for 92751 windows...


  0%|          | 0/698 [00:00<?, ?it/s]

Dataset successfully loaded into RAM. Size: 1.34 GB
[INFO] Val  windows : 20172
[INFO] Test windows : 92751
[INFO] Test windows: 92751
[INFO] Estimated total surgery hours: 1545.8 h

[STEP A] Selecting threshold from M1 validation set using Youden's J...


...

[NOTE] Lead time is estimated from pipeline parameters (uniform distribution over [5, 15] min). For exact per-window lead times, store IOH onset offsets during preprocessing.

[EVALUATING] Transformer — Full signals (D1) ...


  0%|          | 0/363 [00:00<?, ?it/s]

  0%|          | 0/363 [00:00<?, ?it/s]

  0%|          | 0/363 [00:00<?, ?it/s]

[NOTE] Lead time is estimated from pipeline parameters (uniform distribution over [5, 15] min). For exact per-window lead times, store IOH onset offsets during preprocessing.



════════════════════════════════════════════════════════════════════════════════
  FULL RESULTS
════════════════════════════════════════════════════════════════════════════════
  Mamba — Full signals (D1)
  AUROC : 0.7360  (95% CI: 0.7301 – 0.7419)
  AUPRC : 0.1794  (95% CI: 0.1717 – 0.1886)
  Random AUPRC baseline: 0.0657
  AUPRC lift over random: 2.73x

  ── Threshold metrics (threshold = 0.2082) ──
  Sensitivity : 0.7236
  Specificity : 0.6185
  PPV         : 0.1177
  NPV         : 0.9695
  F1-Score    : 0.2025
  TP=4411  TN=53598  FP=33057  FN=1685

  ── Clinical utility ──
  True positive predictions   : 4411
  Mean lead time              : 10.0 ± 2.9 min
  False alarm rate            : 21.38 alarms/hour
  Total surgery hours (test)  : 1545.8 h

  ── Per-seed AUROC / AUPRC ──
  Seed 1: AUROC=0.7321  AUPRC=